In [1]:
import pandas as pd

# 1. Cargar datasets
df_crudo = pd.read_csv('datos_crudos/dataset_laptops_mexico.csv')
df_enriquecido = pd.read_csv('EDA_data_cleaning/catalogo_enriquecido.csv')

# 2. Extraer solo Nombre y URL_Producto, eliminando duplicados de Nombre
urls_df = df_crudo[['Nombre', 'URL_Producto']].dropna().drop_duplicates(subset=['Nombre'], keep='first')

# 3. Eliminar la columna URL_Producto previa si existiera en el enriquecido para evitar duplicados
if 'URL_Producto' in df_enriquecido.columns:
    df_enriquecido = df_enriquecido.drop(columns=['URL_Producto'])

# 4. Cruce (Left Join) conservando todo el catálogo enriquecido
df_final = pd.merge(df_enriquecido, urls_df, on='Nombre', how='left')

# 5. Fallback de seguridad: Si algún producto no tuvo match perfecto, genera una búsqueda en Google/MercadoLibre
df_final['URL_Producto'] = df_final['URL_Producto'].fillna(
    'https://www.google.com/search?q=' + df_final['Nombre'].apply(lambda x: str(x).replace(' ', '+'))
)

# Verificación
recuperadas = df_final['URL_Producto'].notna().sum()
print(f" URLs asociadas correctamente: {recuperadas} de {len(df_final)} laptops.")

# 6. Guardar en ambas ubicaciones
df_final.to_csv('EDA_data_cleaning/catalogo_enriquecido.csv', index=False)
df_final.to_csv('backend/catalogo_enriquecido.csv', index=False)

print(" Catálogo actualizado con éxito en EDA_data_cleaning/ y backend/.")

 URLs asociadas correctamente: 2182 de 2182 laptops.
 Catálogo actualizado con éxito en EDA_data_cleaning/ y backend/.
